# Exercise 05 — Translating with a Pre-trained Transformer

In the previous notebook you built an encoder–decoder Transformer and trained it to translate dates. The models that translate real languages have **exactly the same architecture** — only bigger, and trained for days on millions of sentence pairs. Nobody trains such a model from scratch for an exercise, and you rarely have to: you download one that somebody else has trained, and adapt it to your data if needed.

In this notebook you work with **`Helsinki-NLP/opus-mt-en-fr`**, an English-to-French translation model, using the Hugging Face `transformers` library. You look inside it, translate with it, measure how good it is, look at its attention, and fine-tune it on a new domain: **novels**.

## What you will do
1. **The data** — English–French sentence pairs from novels.
2. **The model** — load it, and compare it with the one you built.
3. **Tokenization** — subwords instead of characters, and preparing sources *and* targets.
4. **Translate and evaluate** — generate translations and score them with **BLEU**.
5. **Cross-attention in a real model** — the same plot as for the dates, for real sentences.
6. **Fine-tuning** — adapt the model to the novels with the `Seq2SeqTrainer`.

## How to work through it
Same as before: run the cells **in order**, fill in every `# TODO`, and make sure each **✅ Check** cell passes before moving on. Tasks that say *Your answer here* want a short written answer.

> **Heads up — this notebook downloads a model and fine-tunes it.** The model (~600 MB) and the dataset (~25 MB) are downloaded in the first cells, so start them early. On a CPU, translating the test set (4.2 and 6.2) takes about half a minute each time and the fine-tuning in Part 6 about 3–5 minutes. Read ahead while they run.

Please update your environment before running this notebook using <code onclick="navigator.clipboard.writeText(this.textContent)" style="cursor:pointer" title="Click to copy">uv sync</code> — this week adds `sentencepiece` (for the tokenizer), `sacrebleu` (for the evaluation) and `accelerate` (for the trainer).

In [ ]:
%matplotlib inline
import tempfile

import matplotlib.pyplot as plt
import sacrebleu
import torch

from datasets import load_dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

torch.manual_seed(0)

# Use the GPU if there is one; a CPU is enough for this notebook.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

## 1. The data: translated novels

[`Helsinki-NLP/opus_books`](https://huggingface.co/datasets/Helsinki-NLP/opus_books) is a collection of copyright-free books that have been aligned sentence by sentence with their translations. We use the English–French part: about 127,000 sentence pairs from novels such as *Jane Eyre*, *The Three Musketeers* and *Twenty Thousand Leagues Under the Sea*.

We only need a small part of it: **200 pairs for testing** and **1,000 pairs for fine-tuning**. That is all a CPU can handle in a few minutes, and it is enough to see what is going on.

*Jörg Tiedemann. 2012. Parallel Data, Tools and Interfaces in OPUS. LREC 2012.*

In [ ]:
books = load_dataset("Helsinki-NLP/opus_books", "en-fr")["train"]
print(books)

split = books.train_test_split(test_size=200, seed=42)
test_data = split["test"]
train_data = split["train"].select(range(1_000))

print(f"\n{len(train_data):,} pairs for fine-tuning, {len(test_data):,} pairs for testing. One example:")
print(test_data[0])

**1.1 Collect the English sentences and the French reference translations of the test set in two lists of strings.**

Hint: `test_data["translation"]` is a list of dictionaries like `{"en": "...", "fr": "..."}`.

In [ ]:
test_sources = [pair["en"] for pair in test_data["translation"]]
test_references = [pair["fr"] for pair in test_data["translation"]]

for source, reference in zip(test_sources[:3], test_references[:3]):
    print("EN:", source)
    print("FR:", reference, "\n")

In [ ]:
# ✅ Check your lists
assert len(test_sources) == 200 and len(test_references) == 200, "there should be 200 sentences on both sides"
assert all(isinstance(s, str) for s in test_sources + test_references), "both lists should contain plain strings"
assert test_sources[0] == test_data[0]["translation"]["en"] and test_references[0] == test_data[0]["translation"]["fr"]
print("Looks good ✅")

## 2. The model

`AutoTokenizer` and `AutoModelForSeq2SeqLM` download the tokenizer and the model that belong to a checkpoint name and pick the right classes for them. The first call downloads ~600 MB (the weights come in two file formats); after that everything is cached on disk.

(We ask for `attn_implementation="eager"` because the default, faster attention implementation cannot return its attention weights, and we want to look at those in Part 5.)

In [ ]:
MODEL_NAME = "Helsinki-NLP/opus-mt-en-fr"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME, attn_implementation="eager").to(device)

print(model)

The printout shows the structure you built in the previous notebook: an encoder and a decoder, both a stack of layers; in every layer `self_attn` with `q_proj`, `k_proj`, `v_proj` and `out_proj`, in the decoder additionally `encoder_attn` (the cross-attention); two linear layers `fc1` and `fc2`; and the layer norms.

**2.1 How big is it?** Count the parameters of the whole model and of its token embedding.

Hints:
- You have counted parameters before: `sum(p.numel() for p in model.parameters())`.
- `model.get_input_embeddings()` returns the `nn.Embedding` layer.

In [ ]:
n_params = sum(p.numel() for p in model.parameters())
n_embedding_params = sum(p.numel() for p in model.get_input_embeddings().parameters())

config = model.config
print(f"{'':<28}{'opus-mt-en-fr':>15}{'your date model':>18}")
print(f"{'parameters':<28}{n_params:>15,}{183_336:>18,}")
print(f"{'... of which embeddings':<28}{n_embedding_params:>15,}")
print(f"{'vocabulary size':<28}{config.vocab_size:>15,}{40:>18,}")
print(f"{'embedding dimension':<28}{config.d_model:>15,}{64:>18,}")
print(f"{'encoder / decoder layers':<28}{f'{config.encoder_layers} / {config.decoder_layers}':>15}{'2 / 2':>18}")
print(f"{'attention heads':<28}{config.encoder_attention_heads:>15,}{4:>18,}")
print(f"{'feed-forward dimension':<28}{config.encoder_ffn_dim:>15,}{128:>18,}")

In [ ]:
# ✅ Check your numbers
assert 70e6 < n_params < 80e6, "the model should have about 75 million parameters"
assert n_embedding_params == config.vocab_size * config.d_model, "the embedding is one vector of size d_model per token of the vocabulary"
print("Looks good ✅")

**2.2 Which share of the parameters sits in the token embedding?** In your date model the embeddings were a negligible part. Why is that so different here — and what would the share be if the encoder, the decoder and the output layer each had their *own* embedding matrix, as in your model?

---

*Answer:*

$59{,}514 \times 512 = 30{,}471{,}168$ of the 75,133,952 parameters, about **41%**, are the token embedding. All the attention and feed-forward layers of the twelve Transformer layers together have about 44 million.

The size of an embedding matrix is *vocabulary × dimension*. The date model had 40 characters × 64 dimensions = 2,560 parameters per embedding. A real translation model needs a subword vocabulary for two languages (59,514 entries), and each of them is a vector of 512 numbers.

The 41% are already the economical version: this model uses **one** matrix for three jobs — the encoder's input embedding, the decoder's input embedding and the output layer that turns the last decoder state into scores over the vocabulary (*weight tying*; that is why `parameters()` counts it only once). With three separate matrices, as in your date model, the embeddings would have $3 \times 30.5 = 91.4$ million parameters, **two thirds** of a model that would then have 136 million.

---

## 3. Tokenization

Your date model worked on characters, which was fine for a vocabulary of 40 symbols and outputs of 10 characters. For real text, characters make the sequences very long (and attention is quadratic in the length), while whole words give an enormous vocabulary that still misses every rare word and name. Translation models sit in between and use **subwords**: frequent words are one token, rare words are split into several pieces. You will learn how such a vocabulary is built later in the course; here you only use it.

**3.1 Run the cell below** and look at how the sentences are split. The `▁` marks the beginning of a word.

In [ ]:
for text in ["The cat sat on the mat.", "D'Artagnan unhesitatingly counterattacked."]:
    ids = tokenizer(text)["input_ids"]
    print(f"{text}\n  {len(ids)} tokens: {tokenizer.convert_ids_to_tokens(ids)}\n")

For training, the model needs the source **and** the target as token ids. The tokenizer does both in one call: `tokenizer(sources, text_target=targets, ...)` returns

- `input_ids` and `attention_mask` for the source (the attention mask is the **padding mask** from the previous notebooks: 1 for real tokens, 0 for padding),
- `labels`: the token ids of the target.

You do not have to shift anything or add a start token: given `labels`, the model builds the decoder input by shifting them one position to the right itself — the teacher forcing you wrote by hand in the previous notebook.

**3.2 Complete `preprocess_function`.** It receives a *batch* of examples (that is how `Dataset.map(..., batched=True)` calls it) and should tokenize the English sentences as sources and the French sentences as targets, both truncated to `MAX_LENGTH` tokens.

In [ ]:
MAX_LENGTH = 64


def preprocess_function(examples):
    sources = [pair["en"] for pair in examples["translation"]]
    targets = [pair["fr"] for pair in examples["translation"]]
    return tokenizer(sources, text_target=targets, max_length=MAX_LENGTH, truncation=True)


tokenized_train = train_data.map(preprocess_function, batched=True, remove_columns=train_data.column_names)
print(tokenized_train)

In [ ]:
# ✅ Check your preprocessing
assert set(tokenized_train.column_names) == {"input_ids", "attention_mask", "labels"}, "expected the columns input_ids, attention_mask and labels"
example = tokenized_train[0]
assert tokenizer.decode(example["input_ids"], skip_special_tokens=True)[:10] == train_data[0]["translation"]["en"][:10], "input_ids should be the English sentence"
assert tokenizer.decode(example["labels"], skip_special_tokens=True)[:10] == train_data[0]["translation"]["fr"][:10], "labels should be the French sentence"
assert example["labels"][-1] == tokenizer.eos_token_id, "the labels should end with the end-of-sequence token"
assert max(len(ids) for ids in tokenized_train["input_ids"]) <= MAX_LENGTH, "the sources should be truncated to MAX_LENGTH"
print("Looks good ✅")

## 4. Translate and evaluate

`model.generate(...)` is the library's version of your `greedy_decode`: it runs the encoder once and then the decoder step by step until every sentence in the batch has produced `</s>`.

**4.1 Complete `translate`.**

Hints:
- Tokenize the batch with `return_tensors="pt"`, `padding=True`, `truncation=True` and `max_length=128`, and move the result to the device with `.to(device)`.
- `model.generate(**batch, num_beams=1, max_length=128)` returns the generated token ids. `num_beams=1` selects greedy decoding. (This model's default is *beam search* with 4 beams, which is a little better and about four times slower. Decoding strategies are a topic of their own later in the course.)
- `tokenizer.batch_decode(ids, skip_special_tokens=True)` turns a batch of ids back into strings.

In [ ]:
def translate(model, texts, batch_size=16):
    # Translate a list of English sentences; returns a list of French sentences.
    model.eval()
    translations = []
    for i in range(0, len(texts), batch_size):
        batch = tokenizer(texts[i:i + batch_size], return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            generated = model.generate(**batch, num_beams=1, max_length=128)
        translations += tokenizer.batch_decode(generated, skip_special_tokens=True)
    return translations


translate(model, ["Attention is all you need.", "My hovercraft is full of eels."])

In [ ]:
# ✅ Check your translations
result = translate(model, ["Thank you very much.", "The house is small.", "Where is the library?"], batch_size=2)
assert isinstance(result, list) and len(result) == 3 and all(isinstance(t, str) for t in result), "translate should return one string per input sentence"
assert "merci" in result[0].lower() and "maison" in result[1].lower() and "bibliothèque" in result[2].lower(), f"unexpected translations: {result}"
assert "</s>" not in result[0] and "<pad>" not in result[1], "remove the special tokens when decoding"
print(result)
print("Looks good ✅")

### BLEU

To put a number on translation quality we need a metric. The standard one is **BLEU**: it counts how many of the **n-grams** (word sequences of length 1 to 4) of the model's translation also occur in the human reference translation, and punishes translations that are too short. It ranges from 0 to 100, where 100 means identical to the reference. We use the `sacrebleu` library, the reference implementation.

**4.2 Translate the test set and compute the BLEU score.** The translation takes about half a minute on a CPU.

Hint: `sacrebleu.corpus_bleu(hypotheses, [references])` — note the extra brackets: there could be several reference translations per sentence, so the function expects a *list of* reference lists. The result has a `.score`.

In [ ]:
hypotheses_before = translate(model, test_sources)
bleu_before = sacrebleu.corpus_bleu(hypotheses_before, [test_references]).score

print(f"BLEU on the test set: {bleu_before:.1f}")

In [ ]:
# ✅ Check your score
assert len(hypotheses_before) == 200, "there should be one translation per test sentence"
assert sacrebleu.corpus_bleu(test_references, [test_references]).score > 99.9, "sanity check failed: the references should score 100 against themselves"
assert 10 < bleu_before < 35, f"the BLEU score should be somewhere around 20, got {bleu_before:.1f}"
print("Looks good ✅")

**4.3 Read some translations next to their references.** (If your French is rusty, paste a few into a translator of your choice — or judge by the structure.)

In [ ]:
for i in range(6):
    print("EN        :", test_sources[i])
    print("model     :", hypotheses_before[i])
    print("reference :", test_references[i], "\n")

**4.4 A BLEU score of about 20 out of 100 sounds terrible. Are the translations terrible?** Based on the examples, explain why the score is so low, and what that tells you about BLEU as a metric.

---

*Answer:*

**No.** The translations are fluent French and, a few slips aside, they say what the English says. The score is low because of what BLEU measures — overlap of word n-grams with **one particular reference** — and because of what these references are:

- **Literary translations are free.** The translator of *The Three Musketeers* wrote *«Non, fût-ce mon frère!»* where the model writes *Pas si c'était mon propre frère!* Both are fine, and they share hardly a word. Human translators restructure sentences, merge and split them, leave things out. (Strictly speaking, many of these books were written in French, so the "reference" is the original and the English is the free translation.)
- **One reference.** There are many good translations of a sentence. BLEU was designed with several references per sentence in mind; here there is one, and every legitimate deviation from it counts as an error.
- **Conventions.** The novels use «guillemets» or a dash for direct speech and the typographic apostrophe `’`; the model writes `"..."` and `'`. For BLEU, those are all mismatches, and each one also breaks the longer n-grams around it.
- **Noise.** The sentence alignment of the corpus is automatic and not perfect. The reference of the fourth example (*«Mon Dieu! dit-elle, saurait-il...»*) is simply a different sentence.

A BLEU score has no absolute meaning. It is useful for **comparing systems on the same test set** — which is what we do in Part 6 — and it has to be read together with actual outputs.

---

## 5. Cross-attention in a real model

In the previous notebook the cross-attention of your date model showed which source characters the decoder looked at for each output character. The pre-trained model gives you the same information: call it with `output_attentions=True` and the output contains `cross_attentions`, a tuple with one tensor per decoder layer, each of shape `(batch_size, num_heads, tgt_len, src_len)`.

**5.1 Complete `show_cross_attention`.** It should run the model on a source sentence and a given translation (teacher forcing, as for the dates), take the cross-attention weights of one layer, **average them over the heads** and plot them.

Hints:
- `tokenizer(source, text_target=target, return_tensors="pt")` gives you `input_ids`, `attention_mask` and `labels` — everything the model needs. Pass them with `**batch`.
- The rows of the weight matrix correspond to the tokens the model predicts, which are the `labels`.

In [ ]:
def show_cross_attention(model, source, target, layer=-1):
    batch = tokenizer(source, text_target=target, return_tensors="pt").to(device)
    model.eval()
    with torch.no_grad():
        outputs = model(**batch, output_attentions=True)

    weights = outputs.cross_attentions[layer][0]  # (num_heads, tgt_len, src_len)
    weights = weights.mean(dim=0).cpu()           # average over the heads -> (tgt_len, src_len)

    source_tokens = tokenizer.convert_ids_to_tokens(batch["input_ids"][0])
    target_tokens = tokenizer.convert_ids_to_tokens(batch["labels"][0])

    plt.figure(figsize=(0.6 * len(source_tokens) + 2, 0.4 * len(target_tokens) + 1.5))
    plt.imshow(weights, cmap="Blues", vmin=0, aspect="auto")
    plt.xticks(range(len(source_tokens)), source_tokens, rotation=45, ha="right")
    plt.yticks(range(len(target_tokens)), target_tokens)
    plt.xlabel("source (attended to)")
    plt.ylabel("output (being written)")
    plt.title(f"Cross-attention, decoder layer {layer}, averaged over the heads")
    plt.tight_layout()
    plt.show()


source = "The quick brown fox jumps over the lazy dog."
target = translate(model, [source])[0]
print(source, "->", target)

show_cross_attention(model, source, target)

**5.2 Try other sentences and other layers** (`layer=0` is the first decoder layer, `layer=-1` the last of the six). Sentences where French orders the words differently are the most interesting, for example adjectives (*the European economic area*) or object pronouns (*I gave it to him*).

In [ ]:
source = "I have never seen the European economic area."
show_cross_attention(model, source, translate(model, [source])[0])

**5.3 Describe what you see.** Is there a diagonal? Where is it broken, and why? Which source tokens collect attention without "deserving" it? Would you call these plots an *explanation* of how the model translates?

---

*Answer:*

There is a rough **diagonal**: English and French mostly put things in the same order, so the decoder moves through the source from left to right.

It is broken exactly where the two languages disagree. *European economic area* becomes *espace économique européen*: French puts the adjectives after the noun, and the plot shows a clear **anti-diagonal** for these three words — `espace` looks at `area`, `européen` at `European`. For *never seen* → *n'ai jamais vu*, the negation is split in two in French: `jamais` looks at `never`, and so, more weakly, does `n`. The article `l'` looks at `the`, but also ahead at `area`: which form of the article is needed depends on the noun that follows.

The column of **`</s>`** (and to a lesser extent the full stop) collects weight in almost every row. These tokens carry no content to translate. They serve as a kind of default position, where a head puts its weight when it needs nothing in particular from the source — a common pattern in trained Transformers.

As an **explanation**, the plot has to be taken with care. It shows the average over 8 heads of one of 6 layers. And the keys and values are *encoder outputs*, not words: after six layers of self-attention, the vector at the position of `area` contains information about the whole sentence, so "attending to the position of `area`" is not the same as "using the word *area*". Information also reaches the output through the decoder's self-attention and the residual connections. Attention plots show where information *could* have been read, which makes them good for sanity checks (and for finding bugs), but they are no proof of how the model arrived at its output.

---

## 6. Fine-tuning

The model was trained on a huge mix of texts from the web, EU documents, subtitles and more. Novels are a different **domain**: long sentences, dialogue, old-fashioned vocabulary, and their own typographic conventions. **Fine-tuning** means continuing the training on data from the new domain for a short while, with a small learning rate.

The `Seq2SeqTrainer` of the `transformers` library contains the training loop you wrote in the previous notebook (and a lot more: logging, checkpoints, evaluation, mixed precision, multiple GPUs). It needs:

- the **model** and the **training arguments** — the hyperparameters, given below,
- the **training data** — `tokenized_train` from 3.2,
- a **data collator**, which turns a list of examples into a padded batch. `DataCollatorForSeq2Seq` pads the inputs with `<pad>` and the labels with `-100`, the value that PyTorch's cross-entropy loss ignores (your `ignore_index`),
- the tokenizer, passed as `processing_class`.

**6.1 Create the trainer and start the training.** One epoch over the 1,000 sentence pairs takes about 3–5 minutes on a CPU. The training loss is printed every 25 steps.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir=tempfile.mkdtemp(),  # required, but we do not save anything
    num_train_epochs=1,
    per_device_train_batch_size=8,
    learning_rate=5e-5,
    logging_steps=25,
    save_strategy="no",             # do not write 300 MB checkpoints to disk
    report_to="none",
    dataloader_pin_memory=torch.cuda.is_available(),
    seed=0,
)

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    data_collator=data_collator,
    processing_class=tokenizer,
)

train_result = trainer.train()

In [ ]:
# ✅ Check the training
assert trainer.state.global_step == 125, "one epoch over 1,000 examples with batch size 8 should be 125 steps"
first_loss, last_loss = trainer.state.log_history[0]["loss"], trainer.state.log_history[-2]["loss"]
print(f"Training loss: {first_loss:.3f} (first 25 steps) -> {last_loss:.3f} (last 25 steps)")
assert last_loss < first_loss, "the training loss should go down"
print("Looks good ✅")

**6.2 Translate the test set again with the fine-tuned model** and compare the BLEU scores.

In [ ]:
hypotheses_after = translate(model, test_sources)
bleu_after = sacrebleu.corpus_bleu(hypotheses_after, [test_references]).score

print(f"BLEU before fine-tuning: {bleu_before:.1f}")
print(f"BLEU after fine-tuning : {bleu_after:.1f}\n")

for i in range(6):
    print("EN        :", test_sources[i])
    print("before    :", hypotheses_before[i])
    print("after     :", hypotheses_after[i])
    print("reference :", test_references[i], "\n")

**6.3 What has changed?** Compare the translations before and after, in particular sentences with direct speech. Did the model become a better translator in these few minutes? What did it learn?

---

*Answer:*

*(From our run: BLEU 20.2 → 22.0.)*

Read the pairs side by side and the largest changes are **conventions of the domain**, not translation quality:

- Direct speech now starts with a **dash** (`-- Il est impossible ...`) instead of `"..."`, as in the French editions of these novels.
- `d-Artagnan` has become `d'Artagnan`, and the space before `!` and `?` is gone (`moi !` → `moi!`), as in the corpus.
- The **literary tenses** appear: *s'enfonça* (passé simple), *eût pu* (subjunctive imperfect), *un monsieur* instead of *un gentleman*. That is how French novels are written, and not how web pages and EU documents are written.
- Here and there a real improvement (*ils n'essaient pas* → *ils ne tenteraient pas* fixes a wrong tense), and here and there a new mistake (*ne lui déplaisait pas* → *ne lui déplaît pas*).

The model did not become a better translator in three minutes and 1,000 sentences — everything it knows about English and French comes from pre-training. Fine-tuning has **shifted its style** towards the new domain, and BLEU rewards that, because every matching dash and apostrophe is a matching n-gram. This is typical: a small amount of in-domain data is enough to adapt *how* a pre-trained model says things.

Two caveats. With 200 test sentences a difference of two BLEU points is at the edge of what you can measure; a serious comparison needs a larger test set and a significance test. And we have only looked at the new domain: train longer or with a larger learning rate, and the model starts to **forget** what it could do before — translate a few modern, non-literary sentences with the fine-tuned model to see whether the dashes have started to show up where they do not belong.

---

## Take-aways

- The models behind real translation systems have the architecture you built yourself in the previous notebook. What separates them from your date model is scale: the vocabulary, the width, the depth, and above all the training data.
- A pre-trained model plus a small amount of in-domain data goes a long way. Fine-tuning mostly *adapts* what is already there — style, conventions, vocabulary — it does not teach a language in five minutes.
- BLEU is cheap and reproducible, but it measures overlap with one particular reference, not quality. Always read some outputs.
- Attention plots are a great debugging and teaching tool, and a rather unreliable explanation.

*References: Tiedemann & Thottingal (2020), "OPUS-MT – Building open translation services for the World". Papineni et al. (2002), "BLEU: a Method for Automatic Evaluation of Machine Translation". Post (2018), "A Call for Clarity in Reporting BLEU Scores".*